### Load PDF file

In [18]:
import os
if os.path.exists("NovaS.pdf"):
    print("File exists")

File exists


### Load Libraries

In [40]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

#### Step 0: Convert PDF into text

In [27]:
text_data = PyPDFLoader("NovaS.pdf").load()
text_data

#  it has two things: metadata and page_content. 
# The page_content is the text of the PDF, and metadata is the metadata of the PDF. We will use the page_content to create embeddings

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by d

In [28]:
# Changing metadata

for page in text_data:
    page.metadata["author"] = "Khushi Purwar"
    
text_data

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Khushi Purwar', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow b

### Step 1: Split into chunks

In [29]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, 
    chunk_overlap=20,
)

chunks = splitter.split_documents(text_data)
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Khushi Purwar', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Khushi Purwar', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='and technology company that has grown gradually over the years. The organization was'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Khushi Purwar', 'mo

In [30]:
len(chunks)

101

### Step 2: Create Embeddings

In [41]:
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
embedded_chunks = embed_model.embed_documents([chunk.page_content for chunk in chunks])

In [42]:
embedded_chunks[0]

[-0.0033062827,
 0.035918936,
 -0.0137393,
 0.004028839,
 0.015125515,
 0.010078566,
 -0.0044209673,
 -0.014967973,
 0.0037000056,
 -0.047522757,
 -0.021896617,
 0.00642129,
 0.00027209602,
 0.0069578076,
 -0.027033541,
 -0.023173532,
 0.0047868956,
 -0.011122266,
 -0.010654455,
 0.0081751095,
 0.00867588,
 -0.021178868,
 0.026159467,
 -0.02663848,
 -0.017010845,
 0.019037284,
 0.018712917,
 -0.025052315,
 0.0012189589,
 0.124226585,
 -0.017574808,
 -0.013111162,
 -0.02303808,
 -0.005200382,
 0.011130028,
 0.0033236952,
 -0.0011170376,
 0.0067250915,
 0.0073606456,
 -0.00222271,
 0.007728906,
 -0.002627778,
 -0.006169913,
 0.009941694,
 -0.011358369,
 0.009541897,
 -0.025024673,
 0.0050621177,
 0.003523619,
 0.0064807185,
 -0.01480931,
 0.0064702085,
 0.01652107,
 -0.027137522,
 -0.010354201,
 -0.00062796206,
 0.0073189815,
 -0.0013102224,
 0.008550668,
 -0.022700176,
 -0.023859581,
 0.00978612,
 0.0295669,
 0.011653918,
 0.02151272,
 -0.011796892,
 0.009242392,
 -0.010659098,
 -0.0111

In [43]:
len(embedded_chunks[0])

3072

### Instead of above step, we can merge creating and storing steps into 1 step

### Step 2&3: Create Embedding and store them into VectorDB

In [49]:
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [ ]:
chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db") 
# created the database in the local directory. The database is created in the form of a folder called chroma_db. 
# This folder contains the embeddings of the chunks. We can use this database to retrieve the chunks based on the query.

### Step 4: Connection & Retrieval

In [51]:
chroma_db_conn = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

C:\Users\v-kpurwar\AppData\Local\Temp\ipykernel_131136\1337055959.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_conn = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [57]:
chroma_db_conn.similarity_search("How many projects are completed by 2017?", k=3)

[Document(metadata={'creator': 'Microsoft® Word for Microsoft 365', 'moddate': '2026-03-31T11:24:15-03:00', 'author': 'Khushi Purwar', 'source': 'NovaS.pdf', 'producer': 'Microsoft® Word for Microsoft 365', 'page_label': '1', 'creationdate': '2026-03-31T11:24:15-03:00', 'page': 0, 'total_pages': 3}, page_content='the end of 2017, the organization had already completed more than ten small projects. This'),
 Document(metadata={'page': 0, 'total_pages': 3, 'source': 'NovaS.pdf', 'moddate': '2026-03-31T11:24:15-03:00', 'creationdate': '2026-03-31T11:24:15-03:00', 'creator': 'Microsoft® Word for Microsoft 365', 'author': 'Khushi Purwar', 'page_label': '1', 'producer': 'Microsoft® Word for Microsoft 365'}, page_content='the end of 2017, the organization had already completed more than ten small projects. This'),
 Document(metadata={'creationdate': '2026-03-31T11:24:15-03:00', 'creator': 'Microsoft® Word for Microsoft 365', 'author': 'Khushi Purwar', 'moddate': '2026-03-31T11:24:15-03:00', 'p

### Step 5: LLM & Answer Generation

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [ ]:
llm.invoke("How many projects are completed by 2017?")
#  because does not have access to the database, it will not be able to answer the question. 
# We need to use the retrieved chunks to answer the question. We will use the retrieved chunks to answer the question.

AIMessage(content='I cannot answer this question without more information.\n\nPlease provide the data or context that describes the projects and their completion dates.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f3214-163b-7f02-bb77-034cebfe0f17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 200, 'total_tokens': 213, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 175}})

In [62]:
user_query = "How many projects are completed by 2017?"
relevant_chunks = chroma_db_conn.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(relevant_chunks):
    rel_chunks_content.append(chunk.page_content)
    
str(rel_chunks_content)

"['the end of 2017, the organization had already completed more than ten small projects. This', 'the end of 2017, the organization had already completed more than ten small projects. This', 'on improving the quality of its work instead of simply increasing the number of projects.']"

In [63]:
llm.invoke(f"Answer the question: {user_query} based on the following context: {str(rel_chunks_content)}")

AIMessage(content='By the end of 2017, the organization had completed **more than ten small projects**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f3217-d9fd-7553-8628-a12992bb640d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 208, 'total_tokens': 295, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 187}})